# 8.5 UMAP (Uniform Manifold Approximation and Projection)

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 8: Dimensionality Reduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-08-dimensionality-reduction/08.05_umap.ipynb)

Companion code for Section 8.5 of the book. Run the cells in order from top to bottom.

**Example 8.5.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits, load_breast_cancer, make_swiss_roll
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import time
import warnings; warnings.filterwarnings('ignore')

# Install: pip install umap-learn
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print('UMAP not available. Install with: pip install umap-learn')

# ── UMAP on Digits dataset ────────────────────────────────────────
digits = load_digits()
X = digits.data
y = digits.target
X_sc = StandardScaler().fit_transform(X)

palette_10 = plt.cm.tab10(np.linspace(0,1,10))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('UMAP Analysis - Hyperparameter Effects and Comparisons',
            fontsize=14, fontweight='bold')

if UMAP_AVAILABLE:
    # ── Effect of n_neighbors ─────────────────────────────────────
    for idx, n_neighbors in enumerate([5, 15, 30, 50]):
        ax = axes[0, idx]
        t0 = time.time()
        reducer = umap.UMAP(n_components=2, n_neighbors=n_neighbors,
                            min_dist=0.1, random_state=42)
        Z_umap = reducer.fit_transform(X_sc)
        t1 = time.time()

        for k in range(10):
            mask = y == k
            ax.scatter(Z_umap[mask,0], Z_umap[mask,1], s=8, alpha=0.8,
                      color=palette_10[k], label=str(k))

        ax.set_title(f'n_neighbors={n_neighbors}  ({t1-t0:.1f}s)', fontsize=11)
        ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
        if idx==0: ax.legend(title='Digit', fontsize=7, ncol=2, markerscale=2)
        ax.grid(alpha=0.2)

    # ── Best UMAP ─────────────────────────────────────────────────
    ax = axes[1, 0]
    reducer_best = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    Z_best = reducer_best.fit_transform(X_sc)
    sc = ax.scatter(Z_best[:,0], Z_best[:,1], c=y, cmap='tab10', s=10, alpha=0.8)
    plt.colorbar(sc, ax=ax, label='Digit')
    # Annotate cluster centres
    for k in range(10):
        mask = y==k
        ax.annotate(str(k), (Z_best[mask,0].mean(), Z_best[mask,1].mean()),
                   fontsize=13, fontweight='bold', ha='center', va='center',
                   bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))
    ax.set_title('Best UMAP (n_neighbors=15)'); ax.grid(alpha=0.2)

    # ── Transform new data (UMAP advantage over t-SNE) ────────────
    ax = axes[1, 1]
    # Split into seen and unseen
    X_seen   = X_sc[:1500]
    y_seen   = y[:1500]
    X_unseen = X_sc[1500:]
    y_unseen = y[1500:]

    reducer_partial = umap.UMAP(n_components=2, n_neighbors=15, random_state=42)
    reducer_partial.fit(X_seen)
    Z_seen   = reducer_partial.transform(X_seen)
    Z_unseen = reducer_partial.transform(X_unseen)  # Transform new points!

    ax.scatter(Z_seen[:,0], Z_seen[:,1], c=y_seen, cmap='tab10',
              s=8, alpha=0.5, label='Seen (fit)')
    ax.scatter(Z_unseen[:,0], Z_unseen[:,1], c=y_unseen, cmap='tab10',
              s=30, alpha=1.0, marker='*', label='New (transform only)')
    ax.set_title('UMAP: Transforming New Points\n(impossible with t-SNE!)')
    ax.legend(fontsize=9); ax.grid(alpha=0.2)

    # ── min_dist effect ───────────────────────────────────────────
    for idx, min_dist in enumerate([0.0, 0.5]):
        ax = axes[1, 2+idx]
        reducer_md = umap.UMAP(n_components=2, n_neighbors=15,
                               min_dist=min_dist, random_state=42)
        Z_md = reducer_md.fit_transform(X_sc)
        ax.scatter(Z_md[:,0], Z_md[:,1], c=y, cmap='tab10', s=8, alpha=0.8)
        effect = 'tight clusters' if min_dist==0.0 else 'spread out'
        ax.set_title(f'min_dist={min_dist} ({effect})')
        ax.grid(alpha=0.2)

else:
    # Fallback: show t-SNE in all panels
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    Z_tsne = tsne.fit_transform(X_sc)
    for ax in axes.ravel():
        ax.scatter(Z_tsne[:,0], Z_tsne[:,1], c=y, cmap='tab10', s=8)
        ax.set_title('t-SNE (install umap-learn for UMAP)')

plt.tight_layout()
plt.savefig('umap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# ── UMAP parameter guide ──────────────────────────────────────────
print('=== UMAP Key Parameters ===')
umap_params = {
    'n_neighbors':  '15 (default); controls local vs global balance; 2-200',
    'min_dist':     '0.1 (default); minimum distance between embedded points; 0.0-1.0',
    'n_components': '2 for viz; can be any number for preprocessing',
    'metric':       'euclidean (default); cosine for text; correlation; manhattan',
    'n_epochs':     '200-500; more = better quality but slower',
    'learning_rate':'1.0 (default); rarely needs changing',
}
for p, desc in umap_params.items():
    print(f'  {p:<15}: {desc}')

print('\n=== Speed Comparison: PCA vs t-SNE vs UMAP (1797 × 64) ===')
for name, method in [
    ('PCA(50)',    PCA(n_components=50)),
    ('PCA(2)',     PCA(n_components=2)),
    ('t-SNE',     TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)),
]:
    t0 = time.time()
    method.fit_transform(X_sc)
    print(f'  {name:<12}: {time.time()-t0:.3f}s')

if UMAP_AVAILABLE:
    t0 = time.time()
    umap.UMAP(n_components=2, random_state=42).fit_transform(X_sc)
    print(f'  UMAP        : {time.time()-t0:.3f}s')